In [1]:
import json
import csv
import requests
import socket
import ipaddress
from collections import Counter

INPUT_JSONL = "filtered_traceroutes.jsonl"
OUTPUT_JSONL = "enriched_traceroutes.jsonl"
OUTPUT_CSV = "hop_enrichment.csv"

resolved_ptr_count = 0

asn_distribution = Counter()          # ASN → count
geo_ip_country = Counter()             # country → unique IPs
probe_country_distribution = Counter() # probe country → count

unique_ips_seen = set()
seen_ips = set()

ip_cache = {}
probe_cache = {}

def is_unknown_ip(ip):
    return ip in ("*", None, "")

def is_public_ip(ip):
    try:
        return ipaddress.ip_address(ip).is_global
    except ValueError:
        return False

def get_ptr(ip):
    try:
        return socket.gethostbyaddr(ip)[0]
    except Exception:
        return None

def get_geo(ip):
    try:
        r = requests.get(f"https://ipinfo.io/{ip}/json", timeout=5)
        if r.status_code == 200:
            d = r.json()
            return d.get("country"), d.get("city")
    except Exception:
        pass
    return None, None

def get_asn(ip):
    try:
        url = "https://stat.ripe.net/data/prefix-overview/data.json"
        params = {"resource": ip}
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            asns = r.json().get("data", {}).get("asns", [])
            if asns:
                return asns[0].get("asn")
    except Exception:
        pass
    return None

def get_probe_info(prb_id):
    if prb_id in probe_cache:
        return probe_cache[prb_id]

    try:
        r = requests.get(f"https://atlas.ripe.net/api/v2/probes/{prb_id}/", timeout=5)
        if r.status_code == 200:
            d = r.json()
            info = (d.get("asn_v4"), d.get("country_code"))
            probe_cache[prb_id] = info
            return info
    except Exception:
        pass

    probe_cache[prb_id] = (None, None)
    return None, None

def enrich_ip(ip):
    if is_unknown_ip(ip):
        return {
            "ptr": None,
            "asn": None,
            "country": None,
            "city": None,
            "status": "no_reply"
        }

    if ip in ip_cache:
        return ip_cache[ip]

    if not is_public_ip(ip):
        ip_cache[ip] = {
            "ptr": None,
            "asn": None,
            "country": None,
            "city": None,
            "status": "non_public"
        }
        return ip_cache[ip]

    ptr = get_ptr(ip)
    country, city = get_geo(ip)
    asn = get_asn(ip)

    ip_cache[ip] = {
        "ptr": ptr,
        "asn": asn,
        "country": country,
        "city": city,
        "status": "ok"
    }
    return ip_cache[ip]

with open(INPUT_JSONL) as infile, \
     open(OUTPUT_JSONL, "w") as jsonl_out, \
     open(OUTPUT_CSV, "w", newline="") as csv_out:

    csv_writer = csv.DictWriter(
        csv_out,
        fieldnames=[
            "msm_id", "prb_id", "hop",
            "ip_addr", "ptr", "asn",
            "country", "city", "rtt"
        ]
    )
    csv_writer.writeheader()

    for line in infile:
        doc = json.loads(line)

        prb_id = doc.get("prb_id")
        msm_id = doc.get("msm_id")

        probe_asn, probe_country = get_probe_info(prb_id)

        doc["prb_asn"] = probe_asn
        doc["prb_country"] = probe_country

        if probe_country:
            probe_country_distribution[probe_country] += 1

        for hop in doc.get("result", []):
            # seen_ips = set()

            for r in hop.get("result", []):
                ip = r.get("ip_addr")
                rtt = r.get("rtt")

                info = enrich_ip(ip)

                # Extend hop entry
                r.update(info)

                # Write CSV row

                if is_unknown_ip(ip) or ip in seen_ips:
                    continue

                unique_ips_seen.add(ip)

                # PTR statistics
                if info["ptr"] is not None:
                    resolved_ptr_count += 1
            
                # ASN distribution
                if info["asn"] is not None:
                    asn_distribution[info["asn"]] += 1
            
                # Geolocation summary (unique IPs)
                if info["country"] is not None:
                    geo_ip_country[info["country"]] += 1
                
                seen_ips.add(ip)
                csv_writer.writerow({
                    "msm_id": msm_id,
                    "prb_id": prb_id,
                    "hop": hop.get("hop"),
                    "ip_addr": ip,
                    "ptr": info["ptr"],
                    "asn": info["asn"],
                    "country": info["country"],
                    "city": info["city"],
                    "rtt": rtt
                })

        jsonl_out.write(json.dumps(doc) + "\n")

with open("statistics.txt", "w", encoding="utf-8") as f:
    print("*** STATISTICS ***", file=f)
    print(f"Unique IPs observed: {len(unique_ips_seen)}", file=f)
    print(f"Resolved PTR records: {resolved_ptr_count}", file=f)
    
    print("\n--- ASNs:", file=f)
    for asn, count in asn_distribution.items():
        print(f"  AS{asn}: {count}", file=f)
    
    print("\n--- Top ASNs:", file=f)
    for asn, count in asn_distribution.most_common(10):
        print(f"  AS{asn}: {count}", file=f)
    
    print("\n--- Geolocation (unique IPs by country):", file=f)
    for country, count in geo_ip_country.items():
        print(f"  {country}: {count}", file=f)

    print("\n--- Top Geolocation (unique IPs by country):", file=f)
    for country, count in geo_ip_country.most_common(10):
        print(f"  {country}: {count}", file=f)
    
    print("\n--- Probe distribution by country:", file=f)
    for country, count in probe_country_distribution.items():
        print(f"  {country}: {count}", file=f)

    print("\n--- Top Probe distribution by country:", file=f)
    for country, count in probe_country_distribution.most_common(10):
        print(f"  {country}: {count}", file=f)

stats = {
    "unique_ips": len(unique_ips_seen),
    "resolved_ptr_records": resolved_ptr_count,
    "asn_distribution": dict(asn_distribution),
    "geolocation_by_country": dict(geo_ip_country),
    "probe_distribution_by_country": dict(probe_country_distribution)
}

with open("statistics_traceroutes.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Enrichment done.")

Enrichment done.
